In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import nltk
from collections import Counter
import torch.optim as optim

nltk.download('punkt')

# =========================
# 1. LOAD DATA
# =========================

dataset = load_dataset("squad")
data = dataset["train"]

pairs = [(item["question"], item["answers"]["text"][0]) for item in data]
pairs = pairs[:5000]

print("Total pairs:", len(pairs))

# =========================
# 2. TOKENIZER
# =========================

def tokenize(text):
    return nltk.word_tokenize(text.lower())

# =========================
# 3. VOCAB
# =========================

counter = Counter()
for q, a in pairs:
    counter.update(tokenize(q))
    counter.update(tokenize(a))

PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

vocab_size = 15000
most_common = counter.most_common(vocab_size - 4)

idx2word = [PAD, UNK, SOS, EOS] + [w for w, _ in most_common]
word2idx = {w: i for i, w in enumerate(idx2word)}

print("Vocab:", len(word2idx))

# =========================
# 4. ENCODE
# =========================

MAX_LEN = 30

def encode(text):
    tokens = [SOS] + tokenize(text) + [EOS]
    ids = [word2idx.get(t, word2idx[UNK]) for t in tokens]

    if len(ids) < MAX_LEN:
        ids += [word2idx[PAD]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

# =========================
# 5. DATASET
# =========================

class QADataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, a = self.pairs[idx]
        return torch.tensor(encode(q)), torch.tensor(encode(a))

dataset = QADataset(pairs)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# =========================
# TRANSFORMER (MINI VERSION)
# =========================

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads=4):
        super().__init__()

        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)

        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        N, seq_len, embed_size = x.shape

        V = self.values(x)
        K = self.keys(x)
        Q = self.queries(x)

        V = V.view(N, seq_len, self.heads, self.head_dim)
        K = K.view(N, seq_len, self.heads, self.head_dim)
        Q = Q.view(N, seq_len, self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", Q, K)
        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)

        out = torch.einsum("nhql,nlhd->nqhd", attention, V)

        out = out.reshape(N, seq_len, embed_size)

        out = self.fc_out(out)

        return out

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.attention = MultiHeadAttention(embed_size)
        self.fc = nn.Linear(embed_size, vocab_size)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Sequential(
            nn.Linear(embed_size, embed_size), 
                                nn.ReLU(), 
                                nn.Linear(embed_size, vocab_size)
                               )
        self.pos_encoding = PositionalEncoding(embed_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.dropout(self.attention(x))              
        x = self.fc(x)                     
        return x

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=100):
        super().__init__()

        
        encoding = torch.zeros(max_len, embed_size)

        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_size, 2) * (-torch.log(torch.tensor(10000.0)) / embed_size))

        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term)
        
        self.encoding = encoding.unsqueeze(0)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1), :]

model = TransformerModel(len(word2idx), 64)

# =========================
# 7. TRAINING
# =========================

criterion = nn.CrossEntropyLoss(ignore_index=word2idx[PAD])
optimizer = optim.Adam(model.parameters(), lr=0.0001)

epochs = 10

for epoch in range(epochs):
    total_loss = 0

    for questions, answers in loader:

        optimizer.zero_grad()

        outputs = model(questions)
        outputs = outputs[:, :-1, :]
        target = answers[:, 1:]

        outputs = outputs.reshape(-1, outputs.shape[-1])
        target = target.reshape(-1)

        loss = criterion(outputs, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# =========================
# 8. PREDICTION
# =========================

def predict_sentence(text, max_len=10):
    model.eval()

    encoded = encode(text)
    input_tensor = torch.tensor([encoded])

    result = []

    with torch.no_grad():
        output = model(input_tensor)

        probs = torch.softmax(output[0], dim=1)

        for i in range(max_len):
            predicted_index = torch.argmax(probs[i]).item()
            word = idx2word[predicted_index]

            
            if len(result) == 0 and word in ["and"]:
                continue

            
            if word == "<EOS>":
                break

            if word not in ["<PAD>", "<SOS>"]:
                result.append(word)

    if len(result) == 0:
        return "I don't know yet"

    return " ".join(result)

# =========================
# 9. TEST
# =========================

print("Example Predictions:\n")

print(predict_sentence("Who is he?"))
print(predict_sentence("What is Python?"))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total pairs: 5000
Vocab: 8198
Epoch 1, Loss: 1304.7428
Epoch 2, Loss: 1016.7854
Epoch 3, Loss: 959.2137
Epoch 4, Loss: 940.8998
Epoch 5, Loss: 931.9567
Epoch 6, Loss: 927.4578
Epoch 7, Loss: 921.6627
Epoch 8, Loss: 913.1004
Epoch 9, Loss: 902.5523
Epoch 10, Loss: 895.6028
Example Predictions:

the
the
